In [ ]:
import json
import re
from pathlib import Path

LOG_PATH = Path("../logs/test/2026-09-22_06-20-47.log")
RESULTS_DIR = Path("uninstall_results")
RESULTS_DIR.mkdir(exist_ok=True)
uninstall_pattern = re.compile(
    r"^\s*Uninstall (?P<name>.+) \[(?P<ID>[^\]]+)\] took (?P<duration>\S+) => "
    r"(?P<status>success|failure) \((?P<method>[^)]+)\)\s*$"
)
uninstall_data = []

with LOG_PATH.open(encoding="utf-8") as f:
    for line in f:
        match = uninstall_pattern.match(line)
        if match:
            record = match.groupdict()
            method = record.pop("method")
            uninstall_data.append({
                "ID": record.pop("ID"),
                **record,
                "uninstall_methods": [{"method": method}],
            })

with open("../results/app_2026-09-21.json", encoding="utf-8") as f:
    app_data = json.load(f)

# The log records uninstall attempts; app records provide categories.
categories_by_id = {
    str(app["itemID"]).casefold(): app.get("categories", [])
    for app in app_data
    if "itemID" in app
}

for record in uninstall_data:
    category = categories_by_id.get(str(record.get("ID", "")).casefold(), [])
    methods = record.get("uninstall_methods", [])

    record["uninstall_methods"] = [
        {**method, "category": category} if isinstance(method, dict)
        else {"method": method, "category": category}
        for method in methods
    ]

# with open("uninstall_methods_with_categories.json", "w", encoding="utf-8") as f:
#     json.dump(uninstall_data, f, ensure_ascii=False, indent=2)



## Category totals by uninstall method

Write category counts for each uninstall method to `results/cat_<log timestamp>.json`.

In [2]:
category_counts_by_method = {}

for record in uninstall_data:
    for uninstall_method in record["uninstall_methods"]:
        method = uninstall_method["method"]
        categories = uninstall_method["category"] or ["Uncategorized"]
        method_counts = category_counts_by_method.setdefault(method, {})

        for category in categories:
            method_counts[category] = method_counts.get(category, 0) + 1

category_counts_by_method = {
    method: dict(sorted(category_counts.items()))
    for method, category_counts in sorted(category_counts_by_method.items())
}

output_path = RESULTS_DIR / f"cat_{LOG_PATH.stem}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(category_counts_by_method, f, ensure_ascii=False, indent=2)


## Applications by uninstall method

Write each uninstall method and its application names to `results/apps_<log timestamp>.json`.

In [ ]:
apps_by_method = {}

for record in uninstall_data:
    for uninstall_method in record["uninstall_methods"]:
        method = uninstall_method["method"]
        apps_by_method.setdefault(method, []).append(record["name"])

apps_by_method = {
    method: sorted(app_names)
    for method, app_names in sorted(apps_by_method.items())
}

output_path = RESULTS_DIR / f"apps_{LOG_PATH.stem}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(apps_by_method, f, ensure_ascii=False, indent=2)
